In [2]:
!pip install datasets

Defaulting to user installation because normal site-packages is not writeable
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 6.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 10.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 20.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 50.1 MB/s  0:00:00
Using cached idna-3.20-py3-none-any.whl (69 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 16.0 MB/s  0:00:03m0:00:0100:01
  Attempting uninstall: idnam━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/19 [pyarrow]
    Found existing installation: idna 3.11━━━━━━━━━━━━━━━━━━━━  1/19 [pyarrow]
    Uninstalling idna-3.11:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/19 [pyarrow]
      Successfully uninstalled idna-3.11━━━━━━━━━━━━━━━━━━━━━━  1/19 [pyarrow]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19/19 [datasets]/19 [datasets]ce-hub]


In [1]:
from datasets import load_dataset

ds = load_dataset("PotatoHD/ru-text-corpus")['train'].to_pandas()
ds = ds[ds.src == 'news'].copy()

In [2]:
ds.text.str.len().sum()

np.int64(191773773)

In [3]:
import re
import pandas as pd

def to_lower(text):
    return str(text).lower() if pd.notna(text) else text


def remove_punct(text):
    return re.sub(r"[^\w\s]", " ", str(text), flags=re.UNICODE) if pd.notna(text) else text


def to_tokens(text):
    return re.findall(r"\w+", str(text), flags=re.UNICODE) if pd.notna(text) else []


In [4]:
ds['tokens'] = ds.text.apply(to_lower).apply(remove_punct).apply(to_tokens)
ds.head(5)

,text,src,tokens
17177,Сегодня транспортный налог начисляется в завис...,news,"[сегодня, транспортный, налог, начисляется, в,..."
17178,Словосочетание «музыкальный кинофестиваль» уже...,news,"[словосочетание, музыкальный, кинофестиваль, у..."
17179,После более чем 12-часовых консультаций Совет ...,news,"[после, более, чем, 12, часовых, консультаций,..."
17180,"Жертвами урагана «Агата», обрушившегося на Цен...",news,"[жертвами, урагана, агата, обрушившегося, на, ..."
17181,Решение ограничить рост тарифов естественных м...,news,"[решение, ограничить, рост, тарифов, естествен..."


In [19]:
#Статистика: число слов, размер словаря, двадцать самых частых слов.
print("Число слов", ds.tokens.str.len().sum())
print("Размер словаря", len(ds.tokens.explode().unique()))
print("Двадцать самых частых слов.", ds.tokens.explode().value_counts().head(20))

Число слов 26805885
Размер словаря 508506
Двадцать самых частых слов. tokens
в      1147476
и       677850
на      504462
не      344342
что     307427
с       305267
по      247341
а       142979
из      137719
за      133083
к       118637
как     111675
но      109778
о       104261
это      99392
для      96082
он       95534
его      86559
от       85971
у        79842
Name: count, dtype: int64


In [20]:
ds_train = ds.sample(frac=0.9)
ds_test = ds.drop(ds_train.index)

In [170]:
import random 

class CounterLM:
    
    def __init__(self, n=2):
        self.prob_table = None
        self.n = n

    @staticmethod
    def to_ngram(tokens, n=2):
        tokens_len = len(tokens)
        return [tuple(tokens[i:i+n]) for i in range( tokens_len-n+1)]


    def fit(self, text_df, tokens):
        prob_table = text_df[tokens].apply(self.to_ngram).explode().value_counts().rename("c_vw").to_frame()
        words_freq = text_df[tokens].explode().value_counts()

        prob_table[["v", "w"]] = pd.DataFrame(prob_table.index.tolist(), index=prob_table.index)
        prob_table["c_v"]  = prob_table["v"].map(words_freq)
        prob_table["prob"] = prob_table["c_vw"] / prob_table["c_v"]

        self.prob_table = prob_table.reset_index()

        

    def prob(self, v, w):
        return self.prob_table[self.prob_table.tokens == (v,w)].prob

    def generate(self, start, output_len=30):
        output_text = [start]
        for i in range(output_len):
            #print(output_text)
            probs = self.prob_table[self.prob_table.v == output_text[-1]]
            tokens = probs.w.tolist()
            wth = probs.prob.tolist()

            output_text += random.choices(tokens, wth)
        return ' '.join(output_text)

    def logprob(text):
        pass


In [171]:
lm = CounterLM(n=3)

lm.fit(ds_train, tokens='tokens')



In [172]:
lm.generate(start='анекдот')

'анекдот в венесуэле руководитель малого количества зачетных баллов и медицинского страхования предмета для отдыха все на роль играет федерер был очень много отвлекающих обстоятельств так что тимченко совладельца евросети в пресс'